<a href="https://colab.research.google.com/github/agbizbuz/learning-ai-ds-ml/blob/main/LLM_Course/RAG_Crash_Course_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Crash Course
[source](https://www.youtube.com/watch?v=o126p1QN_RI&t=1406s)

In [ ]:
!pip install langchain langchain-core langchain-community langchain-text-splitters langchain_groq pypdf pymupdf sentence-transformers faiss-cpu chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.5/343.5 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8

In [ ]:
# imports
import kagglehub

import os
from pathlib import Path
from typing import List, Dict, Any
import uuid
import time
import numpy as np

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from sentence_transformers import SentenceTransformer
import chromadb


# to access keys
from google.colab import userdata

In [ ]:
## Data Ingestion

### document data structure


path = kagglehub.dataset_download("youssef19/machine-learning-lectures")

100%|██████████| 145k/145k [00:00<00:00, 618kB/s]

Extracting files...


In [ ]:
# Read all pdfs
def process_all_pdfs(path):
  all_docs = []
  for root, dirs, files in os.walk(path):
    for file in files:
      if file.endswith(".pdf"):
        print(f"Processing {file}")
        try:
          loader = PyMuPDFLoader(os.path.join(root, file))
          docs = loader.load()

          #add source information to metadata
          for doc in docs:
            doc.metadata["source_file"] = f"{root}/{file}"
            doc.metadata["file_type"] = 'pdf'

          all_docs.extend(docs)
          print(f" ✓ Loaded {len(docs)} pages")
        except Exception as e:
          print(f" ✕ Error: {e}")
          pass
  return all_docs

In [ ]:
# process all pdfs in the downloaded data
all_pdf_docs = process_all_pdfs(path)

Processing MachineLearning-Lecture01.pdf
 ✓ Loaded 22 pages
Processing MachineLearning-Lecture02.pdf
 ✓ Loaded 18 pages
Processing MachineLearning-Lecture03.pdf
 ✓ Loaded 16 pages


In [ ]:
# Text splitting to get chunks
def split_docs(docs, chunk_size=1000, chunk_overlap=20):
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap=chunk_overlap,
      length_function=len,
      separators=["\n\n", "\n", ".", " ", ""]
  )
  split_docs =  text_splitter.split_documents(docs)
  print(f"Split {len(docs)} into {len(split_docs)} chunks")

  # show an example chunk
  if split_docs:
    print(f"\nExample chunk:")
    print(f"Content: {split_docs[0].page_content[:200]}...")
    print(f"Metadat: {split_docs[0].metadata}")

  return split_docs

In [ ]:
chunks = split_docs(all_pdf_docs)
print(len(chunks))

Split 56 into 205 chunks

Example chunk:
Content: MachineLearning-Lecture01  
Instructor (Andrew Ng): Okay. Good morning. Welcome to CS229, the machine 
learning class. So what I wanna do today is just spend a little time going over the logistics 
of...
Metadat: {'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2008-07-11T11:25:23-07:00', 'source': '/root/.cache/kagglehub/datasets/youssef19/machine-learning-lectures/versions/1/MachineLearning-Lecture01.pdf', 'file_path': '/root/.cache/kagglehub/datasets/youssef19/machine-learning-lectures/versions/1/MachineLearning-Lecture01.pdf', 'total_pages': 22, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2008-07-11T11:25:23-07:00', 'trapped': '', 'modDate': "D:20080711112523-07'00'", 'creationDate': "D:20080711112523-07'00'", 'page': 0, 'source_file': '/root/.cache/kagglehub/datasets/youssef19/machine-learning-lectures/versions/1/MachineLearnin

In [ ]:
class EmbeddingManager:
  """Handles Document embedding generation using SentenceTransformer"""

  def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
    self.model_name = model_name
    self.model = SentenceTransformer(model_name)

  def generate_embeddings(self, texts: List[str]) -> np.ndarray:
    """Generates embeddings for a list of texts"""
    embeddings = self.model.encode(texts, show_progress_bar=True)
    print(f"Generated embeddings with shape: {embeddings.shape}")
    return embeddings

In [ ]:
embd_manger = EmbeddingManager()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Vector Store
class VectorStore:
  """Handles Vector storage using ChromaDB"""

  def __init__(self, collection_name: str = "pdf_docs", persiste_dir: str = "../data/vector_store"):
    self.collection_name = collection_name
    self.persist_dir = persiste_dir
    self.client = None
    self.collection = None
    self._initialize_store()

  def _initialize_store(self):
    try:
      os.makedirs(self.persist_dir, exist_ok=True)
      self.client = chromadb.PersistentClient(path=self.persist_dir)
      self.collection = self.client.get_or_create_collection(
          name=self.collection_name,
          metadata={"description": "PDF Document embeddings for RAG"}
          )
      print(f"Vector store initialized with collection: {self.collection_name}")
      print(f"Existing documents in collection: {self.collection.count()}")
    except Exception as e:
      print(f"Error initializing vector store: {e}")
      raise

  def add_documents(self, documents: List[Any], embeddings: np.ndarray):
    """Adds documents and their embeddings to the vector store"""
    if (len(documents) != embeddings.shape[0]):
      raise ValueError(f"Number of documents ({len(documents)}) does not match the number of embeddings ({embeddings.shape[0]})")

    print(f"Adding {len(documents)} documents to the vector store...")

    # prepare data for ChromaDB
    ids = []
    metadatas = []
    documents_text = []
    embeddings_list = []

    for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
      doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
      ids.append(doc_id)

      metadata = dict(doc.metadata)
      metadata['doc_index'] = i
      metadata['content_lenght'] = len(doc.page_content)
      metadatas.append(metadata)

      documents_text.append(doc.page_content)
      embeddings_list.append(embedding.tolist())

      # Add to collection
      try:
        self.collection.add(
            ids = ids,
            embeddings = embeddings_list,
            metadatas = metadatas,
            documents = documents_text
        )
        print(f" ✓ Added {len(documents)} documents to the vector store")
        print(f"Total documents in the collection: {self.collection.count()}")
      except Exception as e:
        print(f" ✕ Error adding documents to the vector store: {e}")
        raise



In [ ]:
vec_store = VectorStore()

Vector store initialized with collection: pdf_docs
Existing documents in collection: 0


In [ ]:
texts = [doc.page_content for doc in chunks]

embeddings = embd_manger.generate_embeddings(texts)


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Generated embeddings with shape: (205, 384)


In [ ]:
vec_store.add_documents(chunks, embeddings)

Adding 205 documents to the vector store...
 ✓ Added 205 documents to the vector store
Total documents in the collection: 1
 ✓ Added 205 documents to the vector store
Total documents in the collection: 2
 ✓ Added 205 documents to the vector store
Total documents in the collection: 3
 ✓ Added 205 documents to the vector store
Total documents in the collection: 4
 ✓ Added 205 documents to the vector store
Total documents in the collection: 5
 ✓ Added 205 documents to the vector store
Total documents in the collection: 6
 ✓ Added 205 documents to the vector store
Total documents in the collection: 7
 ✓ Added 205 documents to the vector store
Total documents in the collection: 8
 ✓ Added 205 documents to the vector store
Total documents in the collection: 9
 ✓ Added 205 documents to the vector store
Total documents in the collection: 10
 ✓ Added 205 documents to the vector store
Total documents in the collection: 11
 ✓ Added 205 documents to the vector store
Total documents in the collecti

In [ ]:
class RAGRetriever:
  """Handles query-based retrieval from vector store"""

  def __init__(self, vec_store: VectorStore, embd_mngr: EmbeddingManager):
    self.vec_store = vec_store
    self.embd_mngr = embd_mngr

  def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
    """Retrieves documents from the vector store based on the given query"""

    print(f"Retrieving documents for query: '{query}'")
    print(f"Top K: {top_k}, Score threshold: {score_threshold}")

    # Generate query embedding
    query_embd = self.embd_mngr.generate_embeddings([query])[0]

    # Search in the vector store
    try:
      results = self.vec_store.collection.query(
          query_embeddings = [query_embd.tolist()],
          n_results = top_k
      )

      retrieved_docs = []

      if results["documents"] and results["documents"][0]:
        documents = results["documents"][0]
        metadatas = results["metadatas"][0]
        distances = results["distances"][0]
        ids = results["ids"][0]

        for i, (doc_id, doc, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
          similarity_score = 1 - distance
          retrieved_docs.append({
              "id": doc_id,
              "page_content": doc,
              "metadata": metadata,
              "distance": distance,
              "similarity_score": similarity_score,
              "rank": i + 1
          })

          print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
          return retrieved_docs

      else:
        print("No documents found in the vector store")
        return []
    except Exception as e:
      print(f"Error retrieving documents: {e}")
      raise




In [ ]:
rag_retriever = RAGRetriever(vec_store, embd_manger)

In [ ]:
rag_retriever.retrieve("What visualizations are best for reinforcement learning?")

Retrieving documents for query: 'What visualizations are best for reinforcement learning?'
Top K: 5, Score threshold: 0.0


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_f23c6042_69',
  'page_content': "many, many years to come up with that one line of code, so this is not easy.  \nSo that was unsupervised learning, and then the last of the four major topics I wanna tell \nyou about is reinforcement learning. And this refers to problems where you don't do one-\nshot decision-making. So, for example, in the supervised learning cancer prediction \nproblem, you have a patient come in, you predict that the cancer is malignant or benign. \nAnd then based on your prediction, maybe the patient lives or dies, and then that's it, \nright? So you make a decision and then there's a consequence. You either got it right or \nwrong. In reinforcement learning problems, you are usually asked to make a sequence of \ndecisions over time.  \nSo, for example, this is something that my students and I work on. If I give you the keys \nto an autonomous helicopter — we actually have this helicopter here at Stanford, — how \ndo you write a program to make it fly, 

In [ ]:
rag_retriever.retrieve("What does Knowledge is beautiful say?")

Retrieving documents for query: 'What does Knowledge is beautiful say?'
Top K: 5, Score threshold: 0.0


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_22d73277_51',
  'page_content': "in his eyes — he has this deep appreciation of the truth and beauty in the universe as \nrevealed to him by the math he does.  \nIn this class, I'm not gonna do any truth and beauty. In this class, I'm gonna talk about \nlearning theory to try to convey to you an understanding of how and why learning \nalgorithms work so that we can apply these learning algorithms as effectively as possible.  \nSo, for example, it turns out you can prove surprisingly deep theorems on when you can \nguarantee that a learning algorithm will work, all right? So think about a learning",
  'metadata': {'creationdate': '2008-07-11T11:25:23-07:00',
   'keywords': '',
   'subject': '',
   'file_path': '/root/.cache/kagglehub/datasets/youssef19/machine-learning-lectures/versions/1/MachineLearning-Lecture01.pdf',
   'format': 'PDF 1.4',
   'total_pages': 22,
   'creationDate': "D:20080711112523-07'00'",
   'page': 13,
   'moddate': '2008-07-11T11:25:23-07:00',
   'mo

In [ ]:
rag_retriever.retrieve("What is contour map")

Retrieving documents for query: 'What is contour map'
Top K: 5, Score threshold: 0.0


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_20763df5_98',
  'page_content': 'height of this plot. So the surface represents the function J of theta and the axes of this \nfunction, or the inputs of this function are the parameters theta zero and theta one, written \ndown here below.',
  'metadata': {'trapped': '',
   'creator': 'PScript5.dll Version 5.2.2',
   'file_path': '/root/.cache/kagglehub/datasets/youssef19/machine-learning-lectures/versions/1/MachineLearning-Lecture02.pdf',
   'content_lenght': 204,
   'page': 5,
   'keywords': '',
   'source_file': '/root/.cache/kagglehub/datasets/youssef19/machine-learning-lectures/versions/1/MachineLearning-Lecture02.pdf',
   'author': '',
   'doc_index': 98,
   'file_type': 'pdf',
   'title': '',
   'source': '/root/.cache/kagglehub/datasets/youssef19/machine-learning-lectures/versions/1/MachineLearning-Lecture02.pdf',
   'creationDate': "D:20080711112505-07'00'",
   'producer': 'Acrobat Distiller 8.1.0 (Windows)',
   'creationdate': '2008-07-11T11:25:05-07:00',
   'tot

## Simple RAG pipeline with Groq LLM

In [ ]:
# set the groq env
mykey=userdata.get('GROQ_API')
# Set your GROQ API Key
#GROQ_API_KEY = getpass("🔑 Enter your GROQ API Key: ")
os.environ["GROQ_API_KEY"] = mykey
print("✅ API Key set!")

✅ API Key set!


In [ ]:
llm = ChatGroq(groq_api_key=mykey, model_name="llama-3.3-70b-versatile", temperature=0.1, max_tokens=1024)

In [ ]:
# Simple RAG function
def rag_simple(query, retriever, llm_instance, top_k):
  results = retriever.retrieve(query, top_k=top_k)
  context = "\n\n".join([doc['page_content'] for doc in results]) if results else ""
  if not context:
    return "No relevant context found."

  prompt = f"""Use the following context to answer the question concisely.
            Contex:
            {context}

            Question: {query}

            Answer:"""

  response = llm.invoke([prompt.format(context=context, query=query)])
  return response.content

In [ ]:
def get_answer(question):
  return rag_simple(question, rag_retriever, llm, 5)

print(get_answer("Who is Andrew NG"))

Retrieving documents for query: 'Who is Andrew NG'
Top K: 5, Score threshold: 0.0


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Andrew Ng is the instructor.


In [ ]:
get_answer("What does this course is about?")

Retrieving documents for query: 'What does this course is about?'
Top K: 5, Score threshold: 0.0


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


'This course is about machine learning.'

In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

In [ ]:
# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }



In [ ]:
# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])